# A simple Backtracking Constraint Solver

A `CSP`(short for <em><u>C</u>onstraint <u>S</u>atisfaction <u>P</u>roblem</em>) is defined as a tripel of the form
$$ \langle \texttt{Vars}, \texttt{Values}, \texttt{Constraints} \rangle $$
where 
 * $\texttt{Vars}$ is a set of *variables*,
 * $\texttt{Values}$ is a set of *values* that these variables can take, and 
 * $\texttt{Constraints}$ is a set of formulas describing *conditions* that the variables 
   must satisfy. 
   
To simplify the implementation we assume that the set $\texttt{Values}$ is a set of numbers.

In [ ]:
interface CSP {
    readonly Vars:    readonly string[];
    readonly Values:  readonly number[]; 
    readonly Constrs: readonly string[];
}

An `Assignment` maps **some** variables to values.

In [ ]:
interface Assignment {
    [variableName: string]: number;
}

A `Solution` is an assignment that maps **every** variable to a value.

In [ ]:
interface Solution {
    [variableName: string]: number;
}

An `AnnotatedConstraint` is a pairs consisting of `formula` and the set `vars` of all variables occurring in this formula. 

In [ ]:
interface AnnotatedConstraint {
    readonly formula: string;
    readonly vars:    ReadonlySet<string>;
}

The function `isComplete`  checks whether all variables in the list `vars` of variables have a value assigned in `Solution`.

In [ ]:
function isComplete(assign: Assignment, vars: readonly string[]): assign is Solution {
    return Object.keys(assign).length == vars.length;
}

The function `collectVariables(expr)` takes a string `expr` that can be interpreted as a valid expression as its input and collects all variables occurring in `expr`.
It takes care to remove names that correspond to predefined values or functions. To achieve this, it defines a set of `builtIns` containing reserved names (like `Math`, `true`, `false`, `abs`, `min`) that are not treated as variables, ensuring that only true variable names are returned by `collectVariables`.

In [ ]:
function collectVariables(expr: string): Set<string> {
    // The negative lookbehind expression '(?<!\.)' ensures we skip properties like 
    // 'abs' in 'Math.abs', while '\b' ensures we match whole words.
    const identifierRegex = /(?<!\.)\b[a-zA-Z_][a-zA-Z0-9_]*\b/g;
    const variables   = new Set<string>();
    const coreGlobals = new Set(['Math', 'true', 'false']);
    let match: RegExpExecArray | null;
    while ((match = identifierRegex.exec(expr)) != null) {
        const candidate = match[0];
        // Because 'abs' is skipped by the regex, we only need to filter 'Math'
        if (!coreGlobals.has(candidate)) {
            variables.add(candidate);
        }
    }
    return variables;
}

In [ ]:
const expr = 'Math.abs(x - y) + Math.abs(z1 - z2)';
collectVariables(expr);

The function `evaluateExpression` takes a string expression and a
variable assignment (context), creates a dynamic function, and evaluates the given expression using the provided values.

In [ ]:
function evaluateExpression(expr: string, context: Assignment): boolean {
    const argNames  = Object.keys(context);
    const argValues = Object.values(context);
    try {
        const func = new Function(...argNames, `return (${expr});`);
        const result: unknown = func(...argValues);
        return typeof result == 'boolean' ? result : false;
    } catch (e) {
        return false;
    }
}

The function `isConsistent` takes four arguments:
- `variable` is a variable.
- `value` is a value that is to be assigned to the variable `variable`.
- `Assignment` is a partial variable assignment that does not assign a value for `variable`
  and that is *consistent* with all constraints in the set `Constraints`.
- `Constraints` is a set of annotated logical formulas.

The function checks whether the assignment
$$ \texttt{assignment} \cup \{\texttt{variable} \mapsto \texttt{value}\}$$
violates any of the formulas in `constraints`. It assumes that
`Assignment` is *consistent*, i.e. all constraints that can already be evaluated with `Assignment`
are evaluated as `true`.

In [ ]:
function isConsistent(
    variable: string,
    value: number,
    assignment: Assignment,
    constraints: readonly AnnotatedConstraint[]
): boolean {
    const newAssignment = Object.assign({}, assignment, { [variable]: value });
    const assignedVars = new Set(Object.keys(newAssignment));
    return constraints
        .every(({ formula, vars }) => 
               !(vars.has(variable) && isSubset(vars, assignedVars)) || 
                 evaluateExpression(formula, newAssignment)
              );
}
function isSubset<T>(subset: ReadonlySet<T>, superset: Set<T>): boolean {
    return [...subset].every(elem => superset.has(elem));
}

In [ ]:
function extendAssignment(asgnmnt: Assignment, variable: string, val: number): Assignment {
    return Object.assign({}, asgnmnt, { [variable]: val });
}

Given a **consistent** *partial variable assignment* `assignment` and a constraint satisfaction problem `csp`,
the function `backtrackSearch(assignment, csp)` tries to extend the given assignment recursively and thereby produce a solution of the given CSP.

In [ ]:
function backtrackSearch(
    assignment: Assignment,
    variables:  readonly string[],
    values:     readonly number[],
    constraints: readonly AnnotatedConstraint[]
): Solution | null {   
    if (isComplete(assignment, variables)) {
        return assignment;
    }
    const unassignedVar = variables.find(v => !(v in assignment));
    if (unassignedVar == undefined) {
        return null;
    }
    for (const value of values) {
        if (isConsistent(unassignedVar, value, assignment, constraints)) {       
            const newAssignment = extendAssignment(assignment, unassignedVar, value);
            const result = backtrackSearch(newAssignment, variables, values, constraints);
            if (result != null) {
                return result;
            }
        }
    }
    return null;
}

The input to the function `solve` is a *constraint satisfaction problem*, i.e. a *CSP*.
The function `solve` tries to compute a solution of this problem via *backtracking*.
Its main purpose is to transform the given *CSP* into an *annotated CSP* where all the formulas
are *annotated* with their variables. That is, the third component of the *CSP* is now no longer a set of formulas but rather a set of pairs of the form `(f, V)` where `f` is a formula and `V` is the set of variables occurring in this formula. The function then calls the auxiliary function `backtrackSearch` that recursively solves the *annotated CSP*.

In [ ]:
function solve(csp: CSP): Solution | null {
    const { Vars, Values, Constrs } = csp;
    const annotatedConstraints: AnnotatedConstraint[] = 
          Constrs.map(f => ({formula: f, vars: collectVariables(f)}));
    return backtrackSearch({}, Vars, Values, annotatedConstraints);
}

In [ ]:
const expr = 'x + y == 7 * z';
const context = { 'x': 3, 'y': 4, 'z': 1 };
const result = evaluateExpression(expr, context);
console.log(`Ergebnis für "${expr}":`, result); 